# Station Map

Interactive map of all current Capital Bikeshare reference stations (`raw_data/stations.parquet`), using each station's `lat`/`lon` and `capacity`.

Uses [folium](https://python-visualization.github.io/folium/) (Leaflet.js + OpenStreetMap tiles). Install it first if needed:

```
pip install folium
```

This notebook only depends on `stations.parquet` — no `raw_trips` or crosswalk data required.

In [1]:
from pathlib import Path

import folium
import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

# Notebook lives in notebooks/, so the project root is one level up.
ROOT = Path.cwd().parent
RAW_DATA = ROOT / "raw_data"
STATIONS_PATH = RAW_DATA / "stations.parquet"
MAP_OUTPUT_PATH = Path.cwd() / "station_map.html"

print(f"Stations path: {STATIONS_PATH} (exists={STATIONS_PATH.exists()})")

Stations path: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor\raw_data\stations.parquet (exists=True)


## Load data

Load the station reference table and drop any rows missing coordinates (can't be plotted).

In [2]:
stations = pd.read_parquet(STATIONS_PATH)

missing_coords = stations["lat"].isna() | stations["lon"].isna()
print(f"Loaded {len(stations):,} stations")
print(f"Missing lat/lon: {missing_coords.sum():,}")

stations = stations.loc[~missing_coords].copy()
print(f"Plottable stations: {len(stations):,}")

print("\nCapacity distribution:")
print(stations["capacity"].describe())

Loaded 866 stations
Missing lat/lon: 0
Plottable stations: 866

Capacity distribution:
count    866.000000
mean      17.012702
std        5.994493
min        7.000000
25%       12.000000
50%       15.000000
75%       19.000000
max       55.000000
Name: capacity, dtype: float64


## Build the map

One circle marker per station, centered on the network's mean coordinates. Marker **radius** and **color** both scale with `capacity` (docks at that station) — bigger/warmer-colored markers are higher-capacity stations. Capacity is bucketed into three tiers (low/medium/high, by tercile) for the color scale, since a continuous gradient is harder to read at a glance on a map with hundreds of points.

In [3]:
CAPACITY_COLORS = {"low": "#2c7bb6", "medium": "#fdae61", "high": "#d7191c"}

low_cutoff, high_cutoff = stations["capacity"].quantile([1 / 3, 2 / 3])


def capacity_tier(capacity: float) -> str:
    if capacity <= low_cutoff:
        return "low"
    if capacity <= high_cutoff:
        return "medium"
    return "high"


def marker_radius(capacity: float) -> float:
    # Scale capacity (roughly 7-55 docks) into a readable marker radius in pixels.
    return max(4, min(18, 3 + capacity * 0.35))


center_lat = stations["lat"].mean()
center_lon = stations["lon"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="OpenStreetMap")

for _, station in stations.iterrows():
    tier = capacity_tier(station["capacity"])
    popup_html = (
        f"<b>{station['name']}</b><br>"
        f"short_name: {station['short_name']}<br>"
        f"capacity: {station['capacity']} docks"
    )
    folium.CircleMarker(
        location=[station["lat"], station["lon"]],
        radius=marker_radius(station["capacity"]),
        color=CAPACITY_COLORS[tier],
        fill=True,
        fill_color=CAPACITY_COLORS[tier],
        fill_opacity=0.75,
        weight=1,
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=station["name"],
    ).add_to(m)

folium.LayerControl().add_to(m)

print(f"Plotted {len(stations):,} stations")
print(f"Capacity tiers: low <= {low_cutoff:.0f}, medium <= {high_cutoff:.0f}, high > {high_cutoff:.0f}")
m

Plotted 866 stations
Capacity tiers: low <= 15, medium <= 19, high > 19


**Legend:** 🔵 blue = low capacity, 🟠 orange = medium capacity, 🔴 red = high capacity (tiers are terciles of the actual capacity distribution above, so they adapt automatically as the station list changes). Marker size also grows with capacity. Click a marker for the station name, `short_name`, and exact capacity; hover for just the name.

## Save a static HTML copy

Useful for viewing outside a live notebook kernel, or sharing the map directly.

In [4]:
m.save(str(MAP_OUTPUT_PATH))
print(f"Map saved to: {MAP_OUTPUT_PATH}")

Map saved to: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor\notebooks\station_map.html
